# Technical Assignment (NLP / IR / Gen AI / Semantic Search)
**Objective:**  The purpose of this assignment is to assess your skills in Natural Language
 Processing (NLP), Information Retrieval (IR), Generative AI, and Semantic Search. You will
 need to source an open dataset, perform an in-depth analysis, build a model, and derive
 actionable insights.

 **Assignment Description:** You are tasked with sourcing an open dataset related to textual data
 and using it to address a problem of your choice within the realm of NLP, IR, Gen AI, or
 Semantic Search. The focus can be on tasks such as text classification, summarization, topic
 modeling, question answering, media recommendations, or semantic search.

## Table of Contents
 1. Dataset Sourcing
 2. Problem Definition
 3. Approach and Pipeline
 4. Implementation
 5. Evaluation
 6. Recommendations

### Dataset Sourcing
For this assignment I've selected the MIcrosoft News Dataset (MIND) to build a retrieval-augmented generation (RAG) system. It is a large-scale dataset of 160k English news articles and 15 million user behaviour logs that was curated specifically for recommendation tasks.

Main reasons behind the choice:
- Its rich textual content including title, abstract, body, category and entities, makes it suitable for NLP-related tasks, e.g. semantic search, information retrieval and question answering.
- It is not limited to one specific domain and covers a diversity of topics which allows to build and test a more generalizable system.
- It has user interaction data that can be used to evaluate retrieval quality.
- The use of news articles as entities seems to align well with Dow Jones' focus.

### Problem Definition
Problem: With the current volume of available content and news, finding and processing relevant information can be both extremely time-consuming and overwhelming. And even more so in case of time-sensitive material like news articles. This is a significant problem, given that 
 ○ Clearly define the problem you intend to solve using the chosen dataset.
 ○ Explain why this problem is significant and what impact solving it might have.

### Approach and Pipeline:
 ○ Outline your approach to solving the problem.
 ○ Describe the tools, libraries, and frameworks you plan to use.
 ○ Detail your pipeline, including data preprocessing, feature engineering, model
 building, and evaluation.

### Evaluation:
 ○ Evaluate the performance of your model(s) using appropriate metrics.
 ○ Discuss the strengths and limitations of your approach.

### Recommendations:
 ○ Offer recommendations for potential improvements or next steps.

### Implementation:
○ Implement your pipeline in a Jupyter notebook.
 ○ Ensure that your code is well-documented and includes explanations for each
 step.
 ○ Visualize and explain your results.

In [221]:
import os
import pandas as pd 
import numpy as np
import pinecone
import time
# from unidecode import unidecode
import torch
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import SpladeEncoder
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
import wordninja
import itertools

In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#### Load and preprocess the data

In [2]:
interactions = pd.read_csv('data/train/behaviors.tsv', sep='\t')

In [16]:
news_df = pd.read_csv('data/train/news.tsv', sep='\t', header=None, names=['id', 'category', 'subcategory', 'news_title', 'news_abstract', 'url', 'entities_title', 'entities_abstract'])

In [17]:
news_specialized_df = news_df[news_df['category'].isin(['news', 'finance'])]

In [4]:
interactions.shape

(156964, 5)

#### Build the index

In [6]:
#instantiate Pinecone client
pc = pinecone.Pinecone(api_key=os.getenv('PINECONE_API'), pool_threads=6)

In [10]:
#create index if it doesn't exist
index_name = 'mind-small'
if index_name not in pc.list_indexes().names():
        print(f"Creating {index_name} index.")
        pc.create_index(
          name=index_name,
          metric="dotproduct",
          dimension=768,
          spec=pinecone.ServerlessSpec(
                                    cloud="aws",
                                    region="us-east-1"
                                  )
        )
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)

Creating mind-small index.


In [11]:
#connect to index
index = pc.Index(index_name)
print(f"Connected to {index_name}.")

Connected to mind-small.


https://huggingface.co/FinLang/finance-embeddings-investopedia

In [15]:
dense_vector_model = SentenceTransformer('FinLang/finance-embeddings-investopedia', device=device)
sparse_vector_model = SpladeEncoder(max_seq_length=512, device=device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\eremi\miniconda3\envs\rag_langchain_ollama\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eremi\.cache\huggingface\hub\models--FinLang--finance-embeddings-investopedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.20k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [24]:
#using lightweight probabilistic segmentation with wordninja
news_specialized_df.loc[:, "subcategory"] = news_specialized_df.loc[:,"subcategory"].apply(lambda x: ' '.join(wordninja.split(x)))

In [65]:
import json
from functools import reduce

In [64]:
def parse_and_filter_entities(entities_string):
    try:
        entities = json.loads(entities_string)
        labels = [item['Label'] for item in filter(lambda a: a['Confidence'] > 0.95, entities)]
    except:
        labels = []
    return labels

In [85]:
#parse entities fields as json and combine those for titles and abstracts in one field
news_specialized_df.loc[:, "entities"] = news_specialized_df.loc[:, ['entities_title', 'entities_abstract']].apply(lambda x: list(set(reduce(lambda a, b: a+b, list(map(parse_and_filter_entities, x))))), axis=1)

In [86]:
data_records = news_specialized_df.to_dict('records')
data = []
for record in data_records:
    data.append({
            "id": record["id"],
            "context": f""""{record["news_title"]}\n{record["news_abstract"]}""",
            "title": record["news_title"],
            "category": record["category"],
            "subcategory": record["subcategory"],
            "entities": record["entities"],
            "sources": record["url"],
        })

In [87]:
#calculate dense and sparse embeddings and load them asyncroniously to the vector database together with associated metadata
async_results = []
it = iter(data)
chunk = tuple(itertools.islice(it, 256))
while chunk:
    ids = [str(x["id"]) for x in chunk]
    contexts = [x["context"] for x in chunk]
    sources = [x['sources'] for x in chunk]
    titles = [x["title"] for x in chunk]
    categories = [x["category"] for x in chunk]
    subcategories = [x["subcategory"] for x in chunk]
    entities = [x["entities"] for x in chunk]
    
    dense_vectors = dense_vector_model.encode(contexts).tolist()
    sparse_vectors = sparse_vector_model.encode_documents(contexts)
    upsert_chunk = []

    for _id, dense_vector, sparse_vector, context, source, title, category, subcategory, entity in zip(ids,
                                                            dense_vectors,
                                                            sparse_vectors,
                                                            contexts,
                                                            sources,
                                                            titles,
                                                            categories,
                                                            subcategories,
                                                            entities
                                                            ):
        metadata = {
            "category": category,
            "subcategory": subcategory,
            "title": title,
            "context": context,
            "entities": entity,
            "source": source 
        }
        upsert_chunk.append(
            {
                "id": _id,
                "values": dense_vector,
                "sparse_values": sparse_vector,
                "metadata": metadata
            }
    )
    async_result = index.upsert(vectors=upsert_chunk, async_req=True)
    async_results.append(async_result.get())
    chunk = tuple(itertools.islice(it, 256))

KeyboardInterrupt: 

In [88]:
##### Candidates Retrieval and Generating Final Recommendations <a class="anchor" id="rag"></a>

In [ ]:
def encode_scale_vectors(query, alpha):
        """Encode sparse and dense vectors, and apply hybrid vector scaling using a convex combination alpha * dense + (1 - alpha) * sparse
        Args:
        alpha (float): 0 == sparse only, 1 == dense only
        """
        #create sparse query vector
        sparse = sparse_vector_model.encode_queries(query)
        #create dense query vector
        dense = dense_vector_model.encode(query).tolist()
        #scale sparse and dense vectors
        hyb_sparse = {
            'indices': sparse['indices'],
            'values':  [v * (1 - alpha) for v in sparse['values']]
        }
        hyb_dense = [v * alpha for v in dense]
        return hyb_dense, hyb_sparse

In [179]:
def retrieve_from_pinecone(index, query, metadata_filter, top_k, alpha=0.5):
    hyb_dense, hyb_sparse = encode_scale_vectors(query=query, alpha=alpha)
    retrieved_docs = index.query(
        top_k=top_k,
        vector=hyb_dense,
        sparse_vector=hyb_sparse,
        include_metadata=True,
        include_values=False,
        filter=metadata_filter
    )
    documents = [Document(page_content=f"""News article: {item['metadata']['context']}\nTopic: {item['metadata']['subcategory']}\nKeywords{item['metadata']['entities']}""",
        metadata={"title": item['metadata']['title'],
        'source': item['metadata']['source']}) for item in retrieved_docs["matches"]]
    _ids = [item['id'] for item in retrieved_docs["matches"]]
    return documents, _ids

In [89]:
#analyse user query and generate retriever filter based on it

In [90]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Optional
from pydantic import BaseModel as PydanticBaseModel
from typing_extensions import Annotated, TypedDict
import pycountry
from pydantic import Field
from tenacity import(
    retry,
    stop_after_attempt,
    wait_random_exponential
)

In [96]:
subcategories = news_specialized_df.subcategory.unique().tolist()

In [97]:
class BaseModel(PydanticBaseModel):
    class Config:
        arbitrary_types_allowed = True

In [153]:
class QueryDetails(BaseModel):
    category: list = Field(description="List of categories of news article the user is looking for",
                          enum=subcategories)
    named_entities: list = Field(description="List of named entitites.")

In [154]:
llm = ChatOllama(model= 'llama3.2', temperature=0.0)
query_structured_llm = llm.with_structured_output(QueryDetails)

In [155]:
query_classification_prompt = ChatPromptTemplate.from_template(
        """
        Analyse user's query and extract th following properties:
        1. List of categories of the news the user is interested in. Choose all the best matches from this list: {subcategories}.
        2. Main named entities and keyword phrases from the user's input.
        Use only data mentioned directly in the input. Output only those properties that you are absolutely sure of based on the user's input.
        Input: {input}
        """
        )

In [156]:
query_classification_chain = query_classification_prompt | query_structured_llm

In [157]:
@retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(3))
def infer_query_metadata(query):
    return query_classification_chain.invoke({'subcategories': subcategories, 'input': query})

In [145]:
user_query = "Whats is going on with Trump?"

In [158]:
user_metadata = infer_query_metadata(query=user_query)

In [164]:
user_metadata

QueryDetails(category=['news politics', 'news us', 'finance news'], named_entities=['Trump'])

In [172]:
user_category = user_metadata.category

In [173]:
user_entities = user_metadata.named_entities

Pinecone doesn´t allow should or boost

In [174]:
metadata_filter = {
            "$and":
            
                 [
                     {"subcategory": {"$in": user_category}},
                     {"entitites": {"$in": user_entities}}]
}

In [175]:
metadata_light_filter = {
            "$or":
                 [
                     {"subcategory": {"$in": user_category}},
                     {"entitites": {"$in": user_entities}}]
}

In [ ]:
#retrieve 3 batches of candidates with different filters
retrieved_documents = []
top_k = 20

#with full filters applied
retrieved_filtered_documents, _ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_filter, top_k=top_k, alpha=0.5)
retrieved_documents.extend(retrieved_filtered_documents)

#with or-filters applied, and don't repeat already retrieved matches
metadata_light_filter["id"] = {"$nin": _ids}
top_k = max(3, top_k - len(_ids))
retrieved_or_filter_documents, add_ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_light_filter, top_k=top_k, alpha=0.5)
_ids.extend(add_ids)
retrieved_documents.extend(retrieved_or_filter_documents)

#without filter and don't repeat already retrieved matches
top_k = max(2, top_k - len(add_ids))
metadata_minimal_filter = {"id": {"$nin": _ids}}
retrieved_minimal_filter_documents, last_ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_minimal_filter, top_k=top_k, alpha=0.5)
retrieved_documents.extend(retrieved_minimal_filter_documents)

In [222]:
rag_llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.2, frequency_penalty = 0.1, api_key=os.getenv('OPENAI_API_KEY'))

In [ ]:
class Articles(BaseModel):
    title: str = Field(description="Title of this particular relevant article")
    source: str = Field(description="Source url of this particular relevant article")
    summary: str = Field(description="Short summary of this particular relevant article")


class RecOutput(BaseModel):
    articles: List[Articles] = Field(description="A list of recommended articles used for the answer.")
    reply: str = Field(description="A conversational output with an answer to the user's question.")

In [246]:
system_prompt = (
        """You are an assistant for question-answering tasks and recommending news articles. Use the following pieces of retrieved context to answer the question. {context}. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
        Return two things:
        1. A conversational output where you give an anwer to the user's question in one short paragraph.
        2. A list of documents, that you used to answer the question. Each document should be represnted as a json with title, source and summary. Make sure that all elements in one json come from the same document.
        """
        )

In [247]:
qa_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system_prompt),
                ("human", "{input}"),
            ]
        )

In [248]:
rag_chain_from_docs = (
                    {
                        "input": lambda x: x["input"],
                        "context": lambda x: x["context"]
                    }
                    | qa_prompt
                    | rag_llm.with_structured_output(RecOutput)
                )

In [249]:
chain = RunnablePassthrough.assign(answer=rag_chain_from_docs)

In [250]:
ans = chain.invoke({"input": user_query, "context": retrieved_documents})

In [251]:
ans['answer'].reply

'President Trump is currently facing significant challenges, including an impeachment inquiry that has caused alarm among Republicans due to shifting explanations from the White House regarding his actions, such as withholding military aid to Ukraine. Additionally, he is facing backlash from his own party over his decision to withdraw troops from Syria, which has led to a potential rebuke from Congressional Republicans. Amidst these controversies, Trump is urging his allies to focus on defending the substance of the impeachment rather than the process.'

In [252]:
ans['answer'].articles

[Articles(title='Shifting explanations from White House alarm some in GOP', source='https://assets.msn.com/labs/mind/AAJ1nOj.html', summary="The shifting White House explanation for President Donald Trump's decision to withhold military aid from Ukraine drew alarm Friday from Republicans as the impeachment inquiry brought a new test of their alliance."),
 Articles(title='Furious Republicans prepare to rebuke Trump on Syria', source='https://assets.msn.com/labs/mind/AAIJ9D6.html', summary='Congressional Republicans appear poised to hand President Trump a stinging rebuke of his Turkey and Syria policy when lawmakers return to Washington this week.'),
 Articles(title='Trump Wants Defense on Substance, Not Process, of Impeachment', source='https://assets.msn.com/labs/mind/AAJt5vU.html', summary='President Trump said he has encouraged his GOP allies to defend him on the substance of the impeachment probe, instead of focusing on criticizing the process.')]